In [0]:
# If needed (run once per cluster)
# %pip install -U "databricks-sdk>=0.68.0"
# dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorSnapshot, MonitorCronSchedule
from databricks.sdk.errors import NotFound, ResourceAlreadyExists

# ---------------- CONFIG ----------------
catalog_name = "chatbot_dev"
bronze_schema = "bronze"

# Where Databricks will write monitoring output tables/artifacts:
output_schema = "chatbot_dev.bronze"
assets_dir = "/healthbot/healthbot/data_quality_monitoring"

# Optional: schedule refresh (Quartz cron). Example: monthly once 2am UTC
schedule = MonitorCronSchedule(quartz_cron_expression="0 0 2 1 * ?", timezone_id='UTC')

# If True, kicks off refresh immediately after creating/confirming monitor
run_refresh_now = True


w = WorkspaceClient()

def q(*parts):
    return ".".join([f"`{p}`" for p in parts])

def ensure_schema(full_schema_name: str):
    # Create schema if missing. If it exists, API will error; we ignore.
    try:
        w.schemas.create(full_name=full_schema_name)
        print(f"Created schema: {full_schema_name}")
    except Exception:
        print(f"Schema exists (or cannot be created): {full_schema_name}")

def monitor_exists(table_fqn: str) -> bool:
    try:
        w.quality_monitors.get(table_name=table_fqn)
        return True
    except NotFound:
        return False
    
# 1) Ensure monitoring output schema exists
ensure_schema(output_schema)

# 2) List all tables in the bronze schema
# You can list via Spark (works well inside notebooks) or via REST/SDK.
# Spark approach (catalog-qualified schema supported in newer Spark versions):
tables = spark.catalog.listTables(f"{catalog_name}.{bronze_schema}")
tables = [t for t in tables if t.tableType.upper() != "VIEW"]

print(f"Found {len(tables)} objects in {catalog_name}.{bronze_schema}")

# 3) Create monitor + refresh for each table
results = []

for t in tables:
    table_fqn = f"{catalog_name}.{bronze_schema}.{t.name}"
    print(f"\n=== {table_fqn} ===")

    try:
        if not monitor_exists(table_fqn):
            w.quality_monitors.create(
                table_name=table_fqn,
                assets_dir=assets_dir,
                output_schema_name=output_schema,
                snapshot=MonitorSnapshot(),   # Snapshot “data profile / quality” monitor
                # schedule=schedule             # remove if you don't want scheduling
            )
            print("Monitor created.")
            action = "CREATED"
        else:
            print("Monitor already exists.")
            action = "EXISTS"

        if run_refresh_now:
            r = w.quality_monitors.run_refresh(table_name=table_fqn)
            refresh_id = getattr(r, "refresh_id", None)
            print(f"Refresh queued. refresh_id={refresh_id}")
            results.append((table_fqn, action, "REFRESH_QUEUED", str(refresh_id)))
        else:
            results.append((table_fqn, action, "NO_REFRESH", ""))

    except ResourceAlreadyExists:
        # rare race condition
        print("Monitor already exists (race).")
        results.append((table_fqn, "EXISTS", "NO_REFRESH", ""))

    except Exception as e:
        print(f"FAILED: {table_fqn}\n{e}")
        results.append((table_fqn, "FAILED", "ERROR", str(e)))

print("\n=== SUMMARY ===")
for row in results:
    print(row)